In [1]:
pip install polars


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade "jinja2>=3.1.5"

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade "numba>=0.61"

  Using cached numba-0.68.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (2.8 kB)
  Using cached llvmlite-0.50.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (5.1 kB)
Using cached numba-0.68.0-cp312-cp312-macosx_12_0_arm64.whl (2.8 MB)
Using cached llvmlite-0.50.0-cp312-cp312-macosx_12_0_arm64.whl (40.5 MB)
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.49.0
    Uninstalling llvmlite-0.49.0:
      Successfully uninstalled llvmlite-0.49.0
  Attempting uninstall: numba
    Found existing installation: numba 0.67.0
    Uninstalling numba-0.67.0:
      Successfully uninstalled numba-0.67.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 3.3.1 requires numba<=0.67.0,>=0.58, but you have numba 0.68.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pymc

  Using cached numba-0.67.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (2.7 kB)
  Using cached llvmlite-0.49.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (5.0 kB)
Using cached numba-0.67.0-cp312-cp312-macosx_12_0_arm64.whl (2.7 MB)
Using cached llvmlite-0.49.0-cp312-cp312-macosx_12_0_arm64.whl (40.5 MB)
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.50.0
    Uninstalling llvmlite-0.50.0:
      Successfully uninstalled llvmlite-0.50.0
  Attempting uninstall: numba
    Found existing installation: numba 0.68.0
    Uninstalling numba-0.68.0:
      Successfully uninstalled numba-0.68.0
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install pytensor

Note: you may need to restart the kernel to use updated packages.


In [6]:
import arviz as az
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import seaborn.objects as so
import numpy as np
import pandas as pd
import polars as pl
import pymc as pm
import pytensor.tensor as pt
import random
import numba, numpy

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_SEED = 42

print(f"Running on PyMC v{pm.__version__}")


Running on PyMC v6.3.2


In [7]:
df = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/MNCAATourneyDetailedResults.csv")
df.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,134,1421,92,1411,84,N,1,32,69,...,31,14,31,17,28,16,15,5,0,22
1,2003,136,1112,80,1436,51,N,0,31,66,...,16,7,7,8,26,12,17,10,3,15
2,2003,136,1113,84,1272,71,N,0,31,59,...,28,14,21,20,22,11,12,2,5,18
3,2003,136,1141,79,1166,73,N,0,29,53,...,17,12,17,14,17,20,21,6,6,21
4,2003,136,1143,76,1301,74,N,1,27,64,...,21,15,20,10,26,16,14,5,8,19


In [8]:
df_city = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/Cities.csv")
df_city.head()

,CityID,City,State
0,4001,Abilene,TX
1,4002,Akron,OH
2,4003,Albany,NY
3,4004,Albuquerque,NM
4,4005,Allentown,PA


In [9]:
df_teamname = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/Mteams.csv")
df_teamname.head()

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026


In [10]:
df_gamecities = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/MGameCities.csv")
df_gamecities = df_gamecities[df_gamecities["CRType"] == "NCAA"]
df_gamecities

,Season,DayNum,WTeamID,LTeamID,CRType,CityID
5263,2010,134,1115,1457,NCAA,4091
5295,2010,136,1124,1358,NCAA,4237
5296,2010,136,1139,1431,NCAA,4305
5297,2010,136,1140,1196,NCAA,4254
5298,2010,136,1242,1250,NCAA,4254
...,...,...,...,...,...,...
86768,2025,146,1120,1277,NCAA,4016
86769,2025,146,1222,1397,NCAA,4161
86785,2025,152,1196,1120,NCAA,4302
86786,2025,152,1222,1181,NCAA,4302


In [11]:
teams = df_teamname[["TeamID", "TeamName"]]
teams


,TeamID,TeamName
0,1101,Abilene Chr
1,1102,Air Force
2,1103,Akron
3,1104,Alabama
4,1105,Alabama A&M
...,...,...
376,1477,East Texas A&M
377,1478,Le Moyne
378,1479,Mercyhurst
379,1480,West Georgia


In [12]:
df_seed = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/MNCAATourneySeeds.csv")
df_seed

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
2689,2026,Z12,1219
2690,2026,Z13,1218
2691,2026,Z14,1244
2692,2026,Z15,1474


In [13]:
df = df.merge(teams.rename(columns={"TeamID": "WTeamID", "TeamName": "WTeamName"}), on="WTeamID", how="left")
df = df.merge(teams.rename(columns={"TeamID": "LTeamID", "TeamName": "LTeamName"}), on="LTeamID", how="left")

In [14]:
seeds = df_seed[["Season", "Seed", "TeamID"]]

df = df.merge(seeds.rename(columns={"TeamID": "WTeamID", "Seed": "WSeed"}), on=["Season", "WTeamID"], how="left")
df = df.merge(seeds.rename(columns={"TeamID": "LTeamID", "Seed": "LSeed"}), on=["Season", "LTeamID"], how="left")

df.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LDR,LAst,LTO,LStl,LBlk,LPF,WTeamName,LTeamName,WSeed,LSeed
0,2003,134,1421,92,1411,84,N,1,32,69,...,28,16,15,5,0,22,UNC Asheville,TX Southern,X16b,X16a
1,2003,136,1112,80,1436,51,N,0,31,66,...,26,12,17,10,3,15,Arizona,Vermont,Z01,Z16
2,2003,136,1113,84,1272,71,N,0,31,59,...,22,11,12,2,5,18,Arizona St,Memphis,Z10,Z07
3,2003,136,1141,79,1166,73,N,0,29,53,...,17,20,21,6,6,21,C Michigan,Creighton,Z11,Z06
4,2003,136,1143,76,1301,74,N,1,27,64,...,26,16,14,5,8,19,California,NC State,W08,W09


In [15]:
df = df.merge(
    df_gamecities[["Season", "DayNum", "WTeamID", "LTeamID", "CityID"]],
    on=["Season", "DayNum", "WTeamID", "LTeamID"],
    how="left",
)
df = df.merge(df_city, on="CityID", how="left")

df.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LStl,LBlk,LPF,WTeamName,LTeamName,WSeed,LSeed,CityID,City,State
0,2003,134,1421,92,1411,84,N,1,32,69,...,5,0,22,UNC Asheville,TX Southern,X16b,X16a,NaN,NaN,NaN
1,2003,136,1112,80,1436,51,N,0,31,66,...,10,3,15,Arizona,Vermont,Z01,Z16,NaN,NaN,NaN
2,2003,136,1113,84,1272,71,N,0,31,59,...,2,5,18,Arizona St,Memphis,Z10,Z07,NaN,NaN,NaN
3,2003,136,1141,79,1166,73,N,0,29,53,...,6,6,21,C Michigan,Creighton,Z11,Z06,NaN,NaN,NaN
4,2003,136,1143,76,1301,74,N,1,27,64,...,5,8,19,California,NC State,W08,W09,NaN,NaN,NaN


In [16]:
df

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LStl,LBlk,LPF,WTeamName,LTeamName,WSeed,LSeed,CityID,City,State
0,2003,134,1421,92,1411,84,N,1,32,69,...,5,0,22,UNC Asheville,TX Southern,X16b,X16a,NaN,NaN,NaN
1,2003,136,1112,80,1436,51,N,0,31,66,...,10,3,15,Arizona,Vermont,Z01,Z16,NaN,NaN,NaN
2,2003,136,1113,84,1272,71,N,0,31,59,...,2,5,18,Arizona St,Memphis,Z10,Z07,NaN,NaN,NaN
3,2003,136,1141,79,1166,73,N,0,29,53,...,6,6,21,C Michigan,Creighton,Z11,Z06,NaN,NaN,NaN
4,2003,136,1143,76,1301,74,N,1,27,64,...,5,8,19,California,NC State,W08,W09,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,2025,146,1120,70,1277,64,N,0,26,61,...,5,5,18,Auburn,Michigan St,Y01,Y02,4016.0,Atlanta,GA
1445,2025,146,1222,69,1397,50,N,0,28,66,...,3,2,9,Houston,Tennessee,X01,X02,4161.0,Indianapolis,IN
1446,2025,152,1196,79,1120,73,N,0,25,53,...,11,3,22,Florida,Auburn,Z01,Y01,4302.0,San Antonio,TX
1447,2025,152,1222,70,1181,67,N,0,23,61,...,7,4,16,Houston,Duke,X01,W01,4302.0,San Antonio,TX


In [17]:
df_final = df[["Season","WScore","LScore","WTeamName","LTeamName","WSeed","LSeed"]]
df_final

,Season,WScore,LScore,WTeamName,LTeamName,WSeed,LSeed
0,2003,92,84,UNC Asheville,TX Southern,X16b,X16a
1,2003,80,51,Arizona,Vermont,Z01,Z16
2,2003,84,71,Arizona St,Memphis,Z10,Z07
3,2003,79,73,C Michigan,Creighton,Z11,Z06
4,2003,76,74,California,NC State,W08,W09
...,...,...,...,...,...,...,...
1444,2025,70,64,Auburn,Michigan St,Y01,Y02
1445,2025,69,50,Houston,Tennessee,X01,X02
1446,2025,79,73,Florida,Auburn,Z01,Y01
1447,2025,70,67,Houston,Duke,X01,W01


In [18]:
df["WSeedNum"] = df["WSeed"].str[1:3].astype(int)
df["LSeedNum"] = df["LSeed"].str[1:3].astype(int)
df

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LPF,WTeamName,LTeamName,WSeed,LSeed,CityID,City,State,WSeedNum,LSeedNum
0,2003,134,1421,92,1411,84,N,1,32,69,...,22,UNC Asheville,TX Southern,X16b,X16a,NaN,NaN,NaN,16,16
1,2003,136,1112,80,1436,51,N,0,31,66,...,15,Arizona,Vermont,Z01,Z16,NaN,NaN,NaN,1,16
2,2003,136,1113,84,1272,71,N,0,31,59,...,18,Arizona St,Memphis,Z10,Z07,NaN,NaN,NaN,10,7
3,2003,136,1141,79,1166,73,N,0,29,53,...,21,C Michigan,Creighton,Z11,Z06,NaN,NaN,NaN,11,6
4,2003,136,1143,76,1301,74,N,1,27,64,...,19,California,NC State,W08,W09,NaN,NaN,NaN,8,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,2025,146,1120,70,1277,64,N,0,26,61,...,18,Auburn,Michigan St,Y01,Y02,4016.0,Atlanta,GA,1,2
1445,2025,146,1222,69,1397,50,N,0,28,66,...,9,Houston,Tennessee,X01,X02,4161.0,Indianapolis,IN,1,2
1446,2025,152,1196,79,1120,73,N,0,25,53,...,22,Florida,Auburn,Z01,Y01,4302.0,San Antonio,TX,1,1
1447,2025,152,1222,70,1181,67,N,0,23,61,...,16,Houston,Duke,X01,W01,4302.0,San Antonio,TX,1,1


In [19]:
df_final = df[["Season","WScore","LScore","WTeamName","LTeamName","WSeedNum","LSeedNum"]]
df_final

,Season,WScore,LScore,WTeamName,LTeamName,WSeedNum,LSeedNum
0,2003,92,84,UNC Asheville,TX Southern,16,16
1,2003,80,51,Arizona,Vermont,1,16
2,2003,84,71,Arizona St,Memphis,10,7
3,2003,79,73,C Michigan,Creighton,11,6
4,2003,76,74,California,NC State,8,9
...,...,...,...,...,...,...,...
1444,2025,70,64,Auburn,Michigan St,1,2
1445,2025,69,50,Houston,Tennessee,1,2
1446,2025,79,73,Florida,Auburn,1,1
1447,2025,70,67,Houston,Duke,1,1


In [20]:
wins = df[["Season", "WSeedNum", "LSeedNum"]].rename(columns={"WSeedNum": "team1", "LSeedNum": "team2"})
wins["outcome"] = 1

losses = df[["Season", "WSeedNum", "LSeedNum"]].rename(columns={"LSeedNum": "team1", "WSeedNum": "team2"})
losses["outcome"] = 0
df_pairs = pd.concat([wins, losses], ignore_index=True)
df_pairs


,Season,team1,team2,outcome
0,2003,16,16,1
1,2003,1,16,1
2,2003,10,7,1
3,2003,11,6,1
4,2003,8,9,1
...,...,...,...,...
2893,2025,2,1,0
2894,2025,2,1,0
2895,2025,1,1,0
2896,2025,1,1,0


In [21]:
len(df_pairs)==2*len(df)

True

In [22]:
wins = df[["Season", "WSeedNum", "LSeedNum", "WTeamName", "LTeamName"]].rename(
    columns={"WSeedNum": "team1", "LSeedNum": "team2", "WTeamName": "team1_name", "LTeamName": "team2_name"}
)
wins["outcome"] = 1

losses = df[["Season", "WSeedNum", "LSeedNum", "WTeamName", "LTeamName"]].rename(
    columns={"LSeedNum": "team1", "WSeedNum": "team2", "LTeamName": "team1_name", "WTeamName": "team2_name"}
)
losses["outcome"] = 0

df_pairs1 = pd.concat([wins, losses], ignore_index=True)
df_pairs1

,Season,team1,team2,team1_name,team2_name,outcome
0,2003,16,16,UNC Asheville,TX Southern,1
1,2003,1,16,Arizona,Vermont,1
2,2003,10,7,Arizona St,Memphis,1
3,2003,11,6,C Michigan,Creighton,1
4,2003,8,9,California,NC State,1
...,...,...,...,...,...,...
2893,2025,2,1,Michigan St,Auburn,0
2894,2025,2,1,Tennessee,Houston,0
2895,2025,1,1,Auburn,Florida,0
2896,2025,1,1,Duke,Houston,0


In [23]:
df_pairs1["winner"] = np.where(df_pairs1["outcome"]== 1, df_pairs1["team1_name"], df_pairs1["team2_name"])

df_pairs1

,Season,team1,team2,team1_name,team2_name,outcome,winner
0,2003,16,16,UNC Asheville,TX Southern,1,UNC Asheville
1,2003,1,16,Arizona,Vermont,1,Arizona
2,2003,10,7,Arizona St,Memphis,1,Arizona St
3,2003,11,6,C Michigan,Creighton,1,C Michigan
4,2003,8,9,California,NC State,1,California
...,...,...,...,...,...,...,...
2893,2025,2,1,Michigan St,Auburn,0,Auburn
2894,2025,2,1,Tennessee,Houston,0,Houston
2895,2025,1,1,Auburn,Florida,0,Florida
2896,2025,1,1,Duke,Houston,0,Houston


In [24]:


def winProbability(teamSeed, opponentSeed):
    if(teamSeed < 1 or teamSeed > 16 or type(teamSeed)!=int):
        print('Please enter valid seed for teamSeed')
        return -1
    if(opponentSeed < 1 or opponentSeed > 16 or type(opponentSeed)!=int):
        print('Please enter valid seed for opponentSeed')
        return -1
    seedSum = opponentSeed + teamSeed
    winProb = opponentSeed / seedSum
    return winProb

def first_round():
    regions = np.empty((4, 8))
    for i in range(4):
        results = np.empty(8)
        for j in range(8):
            if winProbability(j + 1, 16 - j) > random.uniform(0, 1):
                results[j] = int(j + 1)
            else:
                results[j] = int(16 - j)
        regions[i] = results
    return regions

def second_round(first_results):
    regions = np.empty((4, 4))
    for i in range(4):
        second_results = np.empty(4)
        for j in range(4):
            if winProbability(int(first_results[i][j]), int(first_results[i][7 - j])) > random.uniform(0, 1):
                second_results[j] = first_results[i][j]
            else:
                second_results[j] = first_results[i][7 - j]
        regions[i] = second_results
    return regions

def sweet_16(second_results):
    regions = np.empty((4, 2))
    for i in range(4):
        sweet_results = np.empty(2)
        for j in range(2):
            if winProbability(int(second_results[i][j]), int(second_results[i][3 - j])) > random.uniform(0, 1):
                sweet_results[j] = second_results[i][j]
            else:
                sweet_results[j] = second_results[i][3 - j]
        regions[i] = sweet_results
    return regions

def elite_8(sweet_results):
    regions = np.empty(4)
    for i in range(4):
        if winProbability(int(sweet_results[i][0]), int(sweet_results[i][1])) > random.uniform(0, 1):
            regions[i] = sweet_results[i][0]
        else:
            regions[i] = sweet_results[i][1]
    return regions

def final_four(elite_results):
    finals = np.empty(2)
    if winProbability(int(elite_results[0]), int(elite_results[1])) > random.uniform(0, 1):
        finals[0] = elite_results[0]
    else:
        finals[0] = elite_results[1]
    if winProbability(int(elite_results[2]), int(elite_results[3])) > random.uniform(0, 1):
        finals[1] = elite_results[2]
    else:
        finals[1] = elite_results[3]
    return finals

def championship(finals_results):
    if winProbability(int(finals_results[0]), int(finals_results[1])) > random.uniform(0, 1):
        return finals_results[0]
    else:
        return finals_results[1]

def simulate_bracket():
    first = first_round()
    second = second_round(first)
    sweet = sweet_16(second)
    elite = elite_8(sweet)
    finals = final_four(elite)
    champ = championship(finals)
    return champ

In [25]:
def simulate_bracket():
    first = first_round()
    second = second_round(first)
    sweet = sweet_16(second)
    elite = elite_8(sweet)
    finals = final_four(elite)
    champ = championship(finals)
    return champ

# all tournament in one wrap 

In [26]:
simulate_bracket()

np.float64(1.0)

In [27]:
results = [simulate_bracket() for _ in range(100000)]
pd.Series(results).value_counts(normalize=True).sort_index()

# running it 10000 times 
# seeing seed 1 wins 74% of the time

1.0     0.74438
2.0     0.16589
3.0     0.05178
4.0     0.01970
5.0     0.00862
6.0     0.00434
7.0     0.00215
8.0     0.00124
9.0     0.00072
10.0    0.00038
11.0    0.00025
12.0    0.00014
13.0    0.00017
14.0    0.00015
15.0    0.00005
16.0    0.00004
Name: proportion, dtype: float64

In [28]:
champions = df.loc[df.groupby("Season")["DayNum"].idxmax(), ["Season", "WSeedNum"]]
champ_dist = champions["WSeedNum"].value_counts(normalize=True).sort_index()
champ_dist

# the actual results from the given data from 2003-2026
# we can see we are off by about ~6%

WSeedNum
1    0.681818
2    0.090909
3    0.136364
4    0.045455
7    0.045455
Name: proportion, dtype: float64

In [29]:
comparison = pd.DataFrame({"simulated": pd.Series(results).value_counts(normalize=True), "actual": champ_dist}).fillna(0).sort_index()
comparison

,simulated,actual
1.0,0.74438,0.681818
2.0,0.16589,0.090909
3.0,0.05178,0.136364
4.0,0.01970,0.045455
5.0,0.00862,0.000000
6.0,0.00434,0.000000
7.0,0.00215,0.045455
8.0,0.00124,0.000000
9.0,0.00072,0.000000
10.0,0.00038,0.000000


In [30]:
comparison = pd.DataFrame({
    "simulated": pd.Series(results).value_counts(normalize=True),
    "actual": champ_dist,
}).fillna(0).sort_index()
comparison

,simulated,actual
1.0,0.74438,0.681818
2.0,0.16589,0.090909
3.0,0.05178,0.136364
4.0,0.01970,0.045455
5.0,0.00862,0.000000
6.0,0.00434,0.000000
7.0,0.00215,0.045455
8.0,0.00124,0.000000
9.0,0.00072,0.000000
10.0,0.00038,0.000000


In [31]:
comparison [["sim%","actual%"]] = (comparison[["simulated","actual"]]*100).astype(int)
comparison

,simulated,actual,sim%,actual%
1.0,0.74438,0.681818,74,68
2.0,0.16589,0.090909,16,9
3.0,0.05178,0.136364,5,13
4.0,0.01970,0.045455,1,4
5.0,0.00862,0.000000,0,0
6.0,0.00434,0.000000,0,0
7.0,0.00215,0.045455,0,4
8.0,0.00124,0.000000,0,0
9.0,0.00072,0.000000,0,0
10.0,0.00038,0.000000,0,0


In [32]:
 # Bradley Terry

In [33]:
DATA = "/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/"
SEASON = 2025

reg = pd.read_csv(DATA + "MRegularSeasonCompactResults.csv")
games = reg[reg["Season"] == SEASON].copy()

# team1 = smaller TeamID, team2 = larger TeamID
games["team1"] = np.minimum(games["WTeamID"], games["LTeamID"])
games["team2"] = np.maximum(games["WTeamID"], games["LTeamID"])

# outcome = 1 if team1 won
games["outcome"] = (games["WTeamID"] == games["team1"]).astype(int)

# loc from team1's point of view: +1 home, -1 away, 0 neutral
winner_home = games["WLoc"].map({"H": 1, "A": -1, "N": 0})
games["loc"] = np.where(games["outcome"] == 1, winner_home, -winner_home)

print(len(games), "games")
print("team1 win rate:", round(games["outcome"].mean(), 3))

5641 games
team1 win rate: 0.509


In [34]:
team_ids = sorted(set(games["team1"]) | set(games["team2"]))
position = {team: i for i, team in enumerate(team_ids)}

games["pos1"] = games["team1"].map(position)
games["pos2"] = games["team2"].map(position)

print(len(team_ids), "teams")


364 teams


In [35]:
with pm.Model() as bt_model:
    theta = pm.ZeroSumNormal("theta", sigma=1, shape=len(team_ids))
    home_adv = pm.Normal("home_adv", mu=0, sigma=1)

    logit_p = theta[games["pos1"].values] - theta[games["pos2"].values] + home_adv * games["loc"].values
    pm.Bernoulli("outcome", logit_p=logit_p, observed=games["outcome"].values)

    idata = pm.sample(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED)


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [theta, home_adv]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 10 seconds.


In [36]:
summary = az.summary(idata, var_names=["home_adv", "theta"])

print("max r_hat:", summary["r_hat"].max())                  
print("min ess_bulk:", summary["ess_bulk"].min())           
print("home advantage:", summary.loc["home_adv", "mean"])

max r_hat: 1.0054618463151976
min ess_bulk: 5411.969870288603
home advantage: 0.6223162298975421


In [ ]:
theta_draws = idata.posterior["theta"].values.reshape(-1, len(team_ids))   
theta_mean = theta_draws.mean(axis=0)

ratings = pd.DataFrame({"TeamID": team_ids, "theta": theta_mean})
ratings = ratings.merge(teams, on="TeamID").sort_values("theta", ascending=False)
ratings.head(15)

,TeamID,theta,TeamName
17,1120,2.722004,Auburn
89,1196,2.696496,Florida
74,1181,2.580952,Duke
113,1222,2.441248,Houston
283,1397,2.332316,Tennessee
3,1104,2.280588,Alabama
271,1385,2.128992,St John's
168,1277,2.072661,Michigan St
163,1272,2.043070,Memphis
72,1179,2.014163,Drake


In [55]:
ratings

,TeamID,theta,TeamName
17,1120,2.722004,Auburn
89,1196,2.696496,Florida
74,1181,2.580952,Duke
113,1222,2.441248,Houston
283,1397,2.332316,Tennessee
...,...,...,...
351,1468,-2.228412,Bellarmine
229,1341,-2.245469,Prairie View
13,1115,-2.360103,Ark Pine Bluff
47,1154,-2.560906,Citadel


In [46]:
def btWinProbability(teamID, opponentID):
    if teamID not in position:
        print('Please enter valid TeamID for teamID')
        return -1
    if opponentID not in position:
        print('Please enter valid TeamID for opponentID')
        return -1
    diff = theta_draws[:, position[teamID]] - theta_draws[:, position[opponentID]]
    winProb = np.mean(1 / (1 + np.exp(-diff)))
    return winProb

btWinProbability(1120, 1179)   



np.float64(0.6561906624536111)

In [47]:
tour = pd.read_csv(DATA + "MNCAATourneyCompactResults.csv")
tour = tour[(tour["Season"] == SEASON) & (tour["DayNum"] >= 136)]   

seeds_2025 = df_seed[df_seed["Season"] == SEASON]
seed_of = dict(zip(seeds_2025["TeamID"], seeds_2025["Seed"].str[1:3].astype(int)))

p_bt = []
p_seed = []
for _, game in tour.iterrows():
    winner = game["WTeamID"]
    loser = game["LTeamID"]
    p_bt.append(btWinProbability(winner, loser))
    p_seed.append(winProbability(int(seed_of[winner]), int(seed_of[loser])))

p_bt = np.array(p_bt)
p_seed = np.array(p_seed)

print("Seed           log-loss =", round(-np.mean(np.log(p_seed)), 3), "  Brier =", round(np.mean((1 - p_seed) ** 2), 3))
print("Bradley-Terry  log-loss =", round(-np.mean(np.log(p_bt)), 3),   "  Brier =", round(np.mean((1 - p_bt) ** 2), 3))

Seed           log-loss = 0.432   Brier = 0.136
Bradley-Terry  log-loss = 0.522   Brier = 0.173


In [48]:
slot_teams = {}                     
for _, row in seeds_2025.iterrows():
    slot = row["Seed"][:3]           
    if slot not in slot_teams:
        slot_teams[slot] = []
    slot_teams[slot].append(row["TeamID"])

In [49]:
def game_winner(teamA, teamB, d):
    teamA = int(teamA)
    teamB = int(teamB)
    diff = theta_draws[d, position[teamA]] - theta_draws[d, position[teamB]]
    winProb = 1 / (1 + np.exp(-diff))
    if winProb > random.uniform(0, 1):
        return teamA
    else:
        return teamB

In [50]:
def build_bracket(d):
    regions = ["W", "X", "Y", "Z"]
    bracket = np.empty((4, 16))
    for i in range(4):
        for s in range(1, 17):
            teams_in_slot = slot_teams[regions[i] + str(s).zfill(2)]
            if len(teams_in_slot) == 1:
                bracket[i][s - 1] = teams_in_slot[0]
            else:                   
                bracket[i][s - 1] = game_winner(teams_in_slot[0], teams_in_slot[1], d)
    return bracket

def first_round(bracket, d):
    regions = np.empty((4, 8))
    for i in range(4):
        for j in range(8):
            regions[i][j] = game_winner(bracket[i][j], bracket[i][15 - j], d)
    return regions

def second_round(first_results, d):
    regions = np.empty((4, 4))
    for i in range(4):
        for j in range(4):
            regions[i][j] = game_winner(first_results[i][j], first_results[i][7 - j], d)
    return regions

def sweet_16(second_results, d):
    regions = np.empty((4, 2))
    for i in range(4):
        for j in range(2):
            regions[i][j] = game_winner(second_results[i][j], second_results[i][3 - j], d)
    return regions

def elite_8(sweet_results, d):
    regions = np.empty(4)
    for i in range(4):
        regions[i] = game_winner(sweet_results[i][0], sweet_results[i][1], d)
    return regions

def final_four(elite_results, d):
    finals = np.empty(2)
    finals[0] = game_winner(elite_results[0], elite_results[1], d)  
    finals[1] = game_winner(elite_results[2], elite_results[3], d)   
    return finals

def championship(finals_results, d):
    return game_winner(finals_results[0], finals_results[1], d)

In [51]:
def simulate_bt_bracket():
    d = random.randint(0, len(theta_draws) - 1)   
    bracket = build_bracket(d)
    first = first_round(bracket, d)
    second = second_round(first, d)
    sweet = sweet_16(second, d)
    elite = elite_8(sweet, d)
    finals = final_four(elite, d)
    champ = championship(finals, d)
    return champ

random.seed(RANDOM_SEED)
bt_results = [simulate_bt_bracket() for _ in range(10000)]

In [52]:
team_name = dict(zip(teams["TeamID"], teams["TeamName"]))


champ_teams = pd.Series(bt_results).map(team_name).value_counts(normalize=True)
print(champ_teams.head(10))


champ_seeds = pd.Series(bt_results).map(seed_of).value_counts(normalize=True).sort_index()
print(champ_seeds)

Florida        0.1273
Auburn         0.1272
Duke           0.1121
Houston        0.0824
Tennessee      0.0650
Alabama        0.0523
St John's      0.0452
Michigan St    0.0367
Drake          0.0334
Memphis        0.0280
Name: proportion, dtype: float64
1     0.4490
2     0.1992
3     0.0462
4     0.0261
5     0.0682
6     0.0253
7     0.0312
8     0.0281
9     0.0086
10    0.0196
11    0.0478
12    0.0326
13    0.0133
14    0.0031
15    0.0011
16    0.0006
Name: proportion, dtype: float64
